In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
bases_utilizadas = 'bases_utilizadas/'
arquivos_gerados = 'arquivos_gerados/'

# 1. Preparação dos dados

## 1.1 Coordenadas dos municípios

In [3]:
# Base da malha municipal do IBGE
gdf = gpd.read_file(bases_utilizadas + 'BR_Municipios_2024', dtype={'CD_MUN': str})

c:\Users\610946784\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver ESRI Shapefile does not support open option DTYPE
  return ogr_read(


In [4]:
gdf_proj = gdf.to_crs(5880)

points = gdf_proj.geometry.representative_point().to_crs(4326)

gdf['lon'] = points.x
gdf['lat'] = points.y

In [5]:
coords = gdf[['CD_MUN','lat','lon']]

In [6]:
gdf = gdf[['CD_MUN', 'NM_MUN', 'NM_UF', 'SIGLA_UF', 'NM_REGIA', 'geometry', 'lon', 'lat']]

In [7]:
# Base de diretórios da Base dos Dados
municipios = pd.read_csv(bases_utilizadas + 'br_bd_diretorios_brasil_municipio.csv', dtype=str)
municipios = municipios[['id_municipio', 'id_municipio_6', 'nome']]

In [8]:
df_mun = municipios.merge(gdf, left_on='id_municipio', right_on='CD_MUN', how='right')

In [9]:
df_cidades = df_mun.copy()
df_cidades.drop(columns=['geometry'], inplace=True)

## 1.2 Base de dados de perícias médicas

In [10]:
pip install pyxlsb

Note: you may need to restart the kernel to use updated packages.


In [11]:
arquivo_xlsb = bases_utilizadas + "CE___Pericias_Municipio_residencia (10).xlsb"
df = pd.read_excel(arquivo_xlsb, skiprows=1, engine="pyxlsb")

df.to_csv(bases_utilizadas + "pericias.csv", index=False, encoding="utf-8")

In [12]:
# Ler a partir daqui caso o passo anterior tenha sido executado

df = pd.read_csv(bases_utilizadas + "pericias.csv", dtype={'Cod. IBGE Município Residência': str,
                                                           'Cód. IBGE municipio APS': str})

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 706379 entries, 0 to 706378
Data columns (total 15 columns):
 #   Column                                   Non-Null Count   Dtype
---  ------                                   --------------   -----
 0   Identificação anonimizada do Requerente  706379 non-null  str  
 1   Cod. IBGE Município Residência           706379 non-null  str  
 2   Município de residência do Interessado   706379 non-null  str  
 3   Cód. IBGE municipio APS                  706379 non-null  str  
 4   Município APS                            706379 non-null  str  
 5   UF da APS                                706379 non-null  str  
 6   2019                                     706379 non-null  int64
 7   2020                                     706379 non-null  int64
 8   2021                                     706379 non-null  int64
 9   2022                                     706379 non-null  int64
 10  2023                                     706379 non-null  int64
 11

In [14]:
# Unir dataframes com código IBGE de 6 dígitos e renomear colunas

def merge_and_rename(df1, df2, left_on, suffix):
    df_merged = df1.merge(df2, left_on=left_on, right_on='id_municipio_6', how='left')
    cols = ['id_municipio', 'id_municipio_6', 'nome', 'CD_MUN', 'NM_MUN', 'NM_UF', 'SIGLA_UF', 'NM_REGIA', 'lon', 'lat']
    for col in cols:
        df_merged.rename(columns={col: col + suffix}, inplace=True)
    return df_merged


df_residencia = merge_and_rename(df, df_cidades, 'Cod. IBGE Município Residência', '_res')
df_final = merge_and_rename(df_residencia, df_cidades, 'Cód. IBGE municipio APS', '_aps')

In [15]:
cols = [
    'Identificação anonimizada do Requerente',
    'CD_MUN_res',
    'NM_MUN_res',
    'SIGLA_UF_res',
    'lon_res',
    'lat_res',
    'CD_MUN_aps',
    'NM_MUN_aps',
    'SIGLA_UF_aps',
    'lon_aps',
    'lat_aps',
    '2019','2020','2021','2022','2023','2024','2025','2026',
    'TOTAL'
]

df_final = df_final[cols]

In [16]:
estados_ne = ['CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA', 'MA', 'PI']

df_final_ne = df_final[df_final['SIGLA_UF_aps'].isin(estados_ne)]

## 1.3 Calcular a distância entre os municípios de residência e da APS

In [17]:
import sys
!{sys.executable} -m pip install haversine

In [18]:
from haversine import haversine

df_final_ne['dist_km'] = df_final_ne.apply(
    lambda row: haversine(
        (row['lat_res'], row['lon_res']),
        (row['lat_aps'], row['lon_aps'])
    ),
    axis=1
)

In [19]:
df_final_ne.to_csv(arquivos_gerados + 'distancia_residencia_pericias_NE.csv', index=False, encoding='UTF-8')